In [91]:
import pandas as pd

In [92]:
# clean benchmark log
import csv

with open('benchmark_log_eredes.csv', 'r') as fin, open('benchmark_log_eredes_cleaned.csv', 'w') as fout:
    data = [row[:10] for row in csv.reader(fin)]
    csv.writer(fout).writerows(data)

In [93]:
data = pd.read_csv('benchmark_log_eredes_cleaned.csv')

# remove unnecessary columns
data.drop("train_steps", axis=1, inplace=True)

# get total cost in scientific notation
data.drop(["total_cost_format", "microgrid"], axis=1, errors='ignore', inplace=True)
data["total_cost_format"] = data["total_cost"].apply(lambda x: f"{x:.4e}")

# convert zipcodes to IDs
unique_zip_codes = data["zip_code"].unique()
microgrid_map = {name: i for i, name in enumerate(unique_zip_codes)}
data["zip_code_idx"] = data["zip_code"].map(microgrid_map)

data.head()

,config_id,total_cost,zip_code,agent,policy_act,policy_net_arch,learning_rate,total_cost_format,zip_code_idx
0,0,1.482524e+07,7830,BasicAgent heuristics,ReLU,"[64, 64]",0.0001,1.4825e+07,0
1,1,1.419058e+06,7830,SB3Agent DQN,ReLU,"[64, 64]",0.0001,1.4191e+06,0
2,2,1.413531e+06,7830,SB3Agent DQN,ReLU,"[64, 64]",0.0005,1.4135e+06,0
3,3,1.382977e+06,7830,SB3Agent DQN,ReLU,"[64, 64]",0.0010,1.3830e+06,0
4,4,1.389145e+06,7830,SB3Agent DQN,ReLU,"[128, 128]",0.0001,1.3891e+06,0


In [94]:
# baseline
data[data["agent"] == "BasicAgent heuristics"].drop_duplicates(subset=["zip_code_idx"])[["total_cost"]]

,total_cost
0,1.482524e+07
37,1.166389e+07
74,8.639400e+06
111,1.353803e+07
148,1.083199e+07
185,1.666375e+07
222,1.396150e+07
259,1.454063e+07
296,1.198375e+07
333,1.260349e+07


In [95]:
heuristic_res = [
    item["total_cost"] for item in data[data["agent"] == "BasicAgent heuristics"]\
    .drop_duplicates(subset=["zip_code_idx"])[["total_cost"]]\
    .to_dict(orient="records")
]

data = data[data["agent"] != "BasicAgent heuristics"].copy()

def add_baseline(row):
    return heuristic_res[row["zip_code_idx"]]

def absolute_improvement(row):
    return heuristic_res[row["zip_code_idx"]] - row["total_cost"]

def add_percentage_improvement(row):
    heuristic_cost = heuristic_res[row["zip_code_idx"]]
    return (heuristic_cost - row["total_cost"]) / heuristic_cost * 100

data["baseline"] = data.apply(add_baseline, axis=1)
data["absolute_improvement"] = data.apply(absolute_improvement, axis=1)
data["percentage_improvement"] = data.apply(add_percentage_improvement, axis=1)

data.head()

,config_id,total_cost,zip_code,agent,policy_act,policy_net_arch,learning_rate,total_cost_format,zip_code_idx,baseline,absolute_improvement,percentage_improvement
1,1,1.419058e+06,7830,SB3Agent DQN,ReLU,"[64, 64]",0.0001,1.4191e+06,0,1.482524e+07,1.340618e+07,90.428096
2,2,1.413531e+06,7830,SB3Agent DQN,ReLU,"[64, 64]",0.0005,1.4135e+06,0,1.482524e+07,1.341171e+07,90.465374
3,3,1.382977e+06,7830,SB3Agent DQN,ReLU,"[64, 64]",0.0010,1.3830e+06,0,1.482524e+07,1.344227e+07,90.671474
4,4,1.389145e+06,7830,SB3Agent DQN,ReLU,"[128, 128]",0.0001,1.3891e+06,0,1.482524e+07,1.343610e+07,90.629865
5,5,1.333669e+06,7830,SB3Agent DQN,ReLU,"[128, 128]",0.0005,1.3337e+06,0,1.482524e+07,1.349157e+07,91.004065


In [96]:
[f"{res:.4e}" for res in heuristic_res]

['1.4825e+07',
 '1.1664e+07',
 '8.6394e+06',
 '1.3538e+07',
 '1.0832e+07',
 '1.6664e+07',
 '1.3961e+07',
 '1.4541e+07',
 '1.1984e+07',
 '1.2603e+07']

In [97]:
# get best general configs according to total_cost
data.sort_values("total_cost").head()

,config_id,total_cost,zip_code,agent,policy_act,policy_net_arch,learning_rate,total_cost_format,zip_code_idx,baseline,absolute_improvement,percentage_improvement
134,134,845403.815286,2715,SB3Agent A2C,Tanh,"[128, 128]",0.0005,8.4540e+05,3,1.353803e+07,1.269262e+07,93.755339
147,147,845403.815286,2715,SB3Agent PPO,Tanh,"[128, 128]",0.0010,8.4540e+05,3,1.353803e+07,1.269262e+07,93.755339
146,146,845403.815286,2715,SB3Agent PPO,Tanh,"[128, 128]",0.0005,8.4540e+05,3,1.353803e+07,1.269262e+07,93.755339
129,129,845403.815286,2715,SB3Agent A2C,ReLU,"[128, 128]",0.0010,8.4540e+05,3,1.353803e+07,1.269262e+07,93.755339
145,145,845403.815286,2715,SB3Agent PPO,Tanh,"[128, 128]",0.0001,8.4540e+05,3,1.353803e+07,1.269262e+07,93.755339


In [98]:
# get best general configs according to total_cost
data.sort_values("percentage_improvement", ascending=False).head()

,config_id,total_cost,zip_code,agent,policy_act,policy_net_arch,learning_rate,total_cost_format,zip_code_idx,baseline,absolute_improvement,percentage_improvement
211,211,999448.763035,4455,SB3Agent PPO,ReLU,"[64, 64]",0.0005,9.9945e+05,5,1.666375e+07,1.566430e+07,94.002259
219,219,999448.763035,4455,SB3Agent PPO,Tanh,"[128, 128]",0.0001,9.9945e+05,5,1.666375e+07,1.566430e+07,94.002259
195,195,999448.763035,4455,SB3Agent DQN,Tanh,"[128, 128]",0.0001,9.9945e+05,5,1.666375e+07,1.566430e+07,94.002259
198,198,999448.763035,4455,SB3Agent A2C,ReLU,"[64, 64]",0.0001,9.9945e+05,5,1.666375e+07,1.566430e+07,94.002259
199,199,999448.763035,4455,SB3Agent A2C,ReLU,"[64, 64]",0.0005,9.9945e+05,5,1.666375e+07,1.566430e+07,94.002259


In [99]:
# for each microgrid, get the best config
best_configs_lst = []
for zip_code in range(len(unique_zip_codes)):
    best_config = data[data["zip_code_idx"] == zip_code].sort_values("total_cost").head(1)
    best_configs_lst += best_config.to_dict(orient="records")

    print(f"Best config for zip_code {best_config['zip_code'].values[0]} (index {zip_code}):")
    print(best_config.to_string(index=False))
    print()

Best config for zip_code 7830 (index 0):
 config_id   total_cost  zip_code        agent policy_act policy_net_arch  learning_rate total_cost_format  zip_code_idx     baseline  absolute_improvement  percentage_improvement
        36 1.324709e+06      7830 SB3Agent PPO       Tanh      [128, 128]          0.001        1.3247e+06             0 1.482524e+07          1.350053e+07               91.064502

Best config for zip_code 4590 (index 1):
 config_id   total_cost  zip_code        agent policy_act policy_net_arch  learning_rate total_cost_format  zip_code_idx     baseline  absolute_improvement  percentage_improvement
        38 1.213088e+06      4590 SB3Agent DQN       ReLU        [64, 64]         0.0001        1.2131e+06             1 1.166389e+07          1.045080e+07               89.599627

Best config for zip_code 4595 (index 2):
 config_id   total_cost  zip_code        agent policy_act policy_net_arch  learning_rate total_cost_format  zip_code_idx     baseline  absolute_improvement

In [100]:
res = pd.DataFrame(best_configs_lst).drop(["config_id", "total_cost_format"], axis=1)

res = res[["zip_code", "agent", "policy_act", "policy_net_arch", "learning_rate", "baseline", "total_cost", "absolute_improvement", "percentage_improvement"]]
res.columns = ["Zip Code", "Agent", "Policy Activation Function", "Policy Network Architecture", "Learning Rate", "Baseline Cost ($)", "Cost ($)", "Absolute Improvement ($)", "Relative Improvement (%)"]
res["Agent"] = res["Agent"].str.replace("SB3Agent", "")
res["Baseline Cost ($)"] = res["Baseline Cost ($)"].apply(lambda x: f"{x:.4e}")
res["Cost ($)"] = res["Cost ($)"].apply(lambda x: f"{x:.4e}")
res["Absolute Improvement ($)"] = res["Absolute Improvement ($)"].apply(lambda x: f"{x:.4e}")
res["Relative Improvement (%)"] = res["Relative Improvement (%)"].apply(lambda x: f"{x:.4f}")

res.to_latex("best_configs.tex", index=False, escape=False)
res

/tmp/ipykernel_68080/1817340525.py:11: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res.to_latex("best_configs.tex", index=False, escape=False)


,Zip Code,Agent,Policy Activation Function,Policy Network Architecture,Learning Rate,Baseline Cost ($),Cost ($),Absolute Improvement ($),Relative Improvement (%)
0,7830,PPO,Tanh,"[128, 128]",0.0010,1.4825e+07,1.3247e+06,1.3501e+07,91.0645
1,4590,DQN,ReLU,"[64, 64]",0.0001,1.1664e+07,1.2131e+06,1.0451e+07,89.5996
2,4595,PPO,Tanh,"[128, 128]",0.0010,8.6394e+06,1.4461e+06,7.1933e+06,83.2613
3,2715,DQN,ReLU,"[64, 64]",0.0001,1.3538e+07,8.4540e+05,1.2693e+07,93.7553
4,4610,PPO,Tanh,"[128, 128]",0.0010,1.0832e+07,1.1231e+06,9.7089e+06,89.6320
5,4455,PPO,Tanh,"[128, 128]",0.0010,1.6664e+07,9.9945e+05,1.5664e+07,94.0023
6,8900,A2C,ReLU,"[128, 128]",0.0010,1.3961e+07,1.1832e+06,1.2778e+07,91.5252
7,1350,PPO,Tanh,"[128, 128]",0.0010,1.4541e+07,1.1052e+06,1.3435e+07,92.3991
8,2650,PPO,Tanh,"[128, 128]",0.0010,1.1984e+07,1.1492e+06,1.0835e+07,90.4101
9,2420,PPO,Tanh,"[128, 128]",0.0010,1.2603e+07,9.8632e+05,1.1617e+07,92.1743
